In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [3]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 68 (delta 12), reused 63 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 150.09 KiB | 7.90 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Task-driven-low-light-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import shutil
import json
import pandas as pd
from IPython.display import display

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
AUDIT_ROOT = KAGGLE_V1_ROOT / "audit"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"

TRAIN_CLEAN_ROOT = KAGGLE_V1_ROOT / f"train_clean_{RUN_TAG}"
VAL_CLEAN_ROOT = KAGGLE_V1_ROOT / f"val_clean_{RUN_TAG}"
TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"

MATERIALIZED_MANIFEST_CSV = AUDIT_ROOT / f"materialized_split_manifest_{RUN_TAG}.csv"
MATERIALIZED_COUNTS_CSV = AUDIT_ROOT / f"materialized_split_counts_{RUN_TAG}.csv"
MATERIALIZED_SUBJECT_COUNTS_CSV = AUDIT_ROOT / f"materialized_split_subject_counts_{RUN_TAG}.csv"

PILOT_ROOT = KAGGLE_V1_ROOT / f"pilot_clean_{RUN_TAG}"
PILOT_TRAIN_ROOT = PILOT_ROOT / "train"
PILOT_VAL_ROOT = PILOT_ROOT / "val"

PILOT_MANIFEST_CSV = AUDIT_ROOT / f"pilot_manifest_{RUN_TAG}.csv"
PILOT_COUNTS_CSV = AUDIT_ROOT / f"pilot_split_counts_{RUN_TAG}.csv"
PILOT_SUBJECTS_JSON = AUDIT_ROOT / f"pilot_train_subjects_{RUN_TAG}.json"

PILOT_CKPT_DIR = CHECKPOINTS_ROOT / f"pilot_clean_detector_{RUN_TAG}_resnet18"
PILOT_BEST_CKPT = PILOT_CKPT_DIR / f"kaggle_v1_clean_pilot_{RUN_TAG}_best.pt"
PILOT_LAST_CKPT = PILOT_CKPT_DIR / f"kaggle_v1_clean_pilot_{RUN_TAG}_last.pt"

VAL_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_pilot_clean_on_val_{RUN_TAG}"
TEST_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_pilot_clean_on_test_{RUN_TAG}"

assert TRAIN_CLEAN_ROOT.exists(), f"Missing: {TRAIN_CLEAN_ROOT}"
assert VAL_CLEAN_ROOT.exists(), f"Missing: {VAL_CLEAN_ROOT}"
assert TEST_CLEAN_ROOT.exists(), f"Missing: {TEST_CLEAN_ROOT}"
assert MATERIALIZED_MANIFEST_CSV.exists(), f"Missing: {MATERIALIZED_MANIFEST_CSV}"

print("TRAIN_CLEAN_ROOT:", TRAIN_CLEAN_ROOT)
print("VAL_CLEAN_ROOT:", VAL_CLEAN_ROOT)
print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("MATERIALIZED_MANIFEST_CSV:", MATERIALIZED_MANIFEST_CSV)
print("PILOT_ROOT:", PILOT_ROOT)
print("PILOT_CKPT_DIR:", PILOT_CKPT_DIR)


TRAIN_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/train_clean_subject_class_balanced_20k
VAL_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/val_clean_subject_class_balanced_20k
TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
MATERIALIZED_MANIFEST_CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/materialized_split_manifest_subject_class_balanced_20k.csv
PILOT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/pilot_clean_subject_class_balanced_20k
PILOT_CKPT_DIR: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/pilot_clean_detector_subject_class_balanced_20k_resnet18


In [6]:
materialized_df = pd.read_csv(MATERIALIZED_MANIFEST_CSV)

print("Loaded rows:", len(materialized_df))
display(materialized_df.head())

display(
    materialized_df.groupby(["split", "class_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "class_name"])
)


Loaded rows: 20000


,split,class_name,subject_id,source_path,dest_path,dest_file_name,width,height,mode
0,test,closed,s0001,/content/drive/MyDrive/task_driven_video_pipel...,/content/drive/MyDrive/task_driven_video_pipel...,s0001_00001_0_0_0_0_0_01.png,86,86,L
1,test,closed,s0001,/content/drive/MyDrive/task_driven_video_pipel...,/content/drive/MyDrive/task_driven_video_pipel...,s0001_00002_0_0_0_0_0_01.png,84,84,L
2,test,closed,s0001,/content/drive/MyDrive/task_driven_video_pipel...,/content/drive/MyDrive/task_driven_video_pipel...,s0001_00003_0_0_0_0_0_01.png,81,81,L
3,test,closed,s0001,/content/drive/MyDrive/task_driven_video_pipel...,/content/drive/MyDrive/task_driven_video_pipel...,s0001_00004_0_0_0_0_0_01.png,78,78,L
4,test,closed,s0001,/content/drive/MyDrive/task_driven_video_pipel...,/content/drive/MyDrive/task_driven_video_pipel...,s0001_00005_0_0_0_0_0_01.png,81,81,L


,split,class_name,count
0,test,closed,8644
1,test,open,8644
2,train,closed,383
3,train,open,383
4,val,closed,973
5,val,open,973


In [7]:
train_df = materialized_df[materialized_df["split"] == "train"].copy()
val_df = materialized_df[materialized_df["split"] == "val"].copy()
test_df = materialized_df[materialized_df["split"] == "test"].copy()

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))

train_subject_summary = (
    train_df
    .pivot_table(index="subject_id", columns="class_name", values="dest_file_name", aggfunc="count", fill_value=0)
    .reset_index()
)

if "open" not in train_subject_summary.columns:
    train_subject_summary["open"] = 0
if "closed" not in train_subject_summary.columns:
    train_subject_summary["closed"] = 0

train_subject_summary["total"] = train_subject_summary["open"] + train_subject_summary["closed"]
train_subject_summary["both_classes"] = (train_subject_summary["open"] > 0) & (train_subject_summary["closed"] > 0)
train_subject_summary = train_subject_summary.sort_values(["total", "subject_id"], ascending=[False, True]).reset_index(drop=True)

display(train_subject_summary.head(30))


Train rows: 766
Val rows: 1946
Test rows: 17288


class_name,subject_id,closed,open,total,both_classes
0,s0003,120,120,240,True
1,s0002,107,107,214,True
2,s0010,68,68,136,True
3,s0005,28,28,56,True
4,s0020,25,25,50,True
5,s0009,24,24,48,True
6,s0007,11,11,22,True


In [8]:
PILOT_TRAIN_SUBJECT_COUNT = 8

pilot_candidates = train_subject_summary[train_subject_summary["both_classes"]].copy()
pilot_train_subjects = pilot_candidates["subject_id"].head(PILOT_TRAIN_SUBJECT_COUNT).tolist()

if len(pilot_train_subjects) < PILOT_TRAIN_SUBJECT_COUNT:
    remaining = train_subject_summary.loc[
        ~train_subject_summary["subject_id"].isin(pilot_train_subjects), "subject_id"
    ].tolist()
    needed = PILOT_TRAIN_SUBJECT_COUNT - len(pilot_train_subjects)
    pilot_train_subjects.extend(remaining[:needed])

pilot_train_subjects = sorted(pilot_train_subjects)

pilot_subject_info = train_subject_summary[
    train_subject_summary["subject_id"].isin(pilot_train_subjects)
].copy().sort_values(["total", "subject_id"], ascending=[False, True])

print("Pilot train subjects:", len(pilot_train_subjects))
print(pilot_train_subjects)
display(pilot_subject_info)


Pilot train subjects: 7
['s0002', 's0003', 's0005', 's0007', 's0009', 's0010', 's0020']


class_name,subject_id,closed,open,total,both_classes
0,s0003,120,120,240,True
1,s0002,107,107,214,True
2,s0010,68,68,136,True
3,s0005,28,28,56,True
4,s0020,25,25,50,True
5,s0009,24,24,48,True
6,s0007,11,11,22,True


In [9]:
pilot_train_df = train_df[train_df["subject_id"].isin(pilot_train_subjects)].copy()
pilot_val_df = val_df.copy()

pilot_subject_payload = {
    "run_tag": RUN_TAG,
    "pilot_train_subject_count": PILOT_TRAIN_SUBJECT_COUNT,
    "pilot_train_subjects": pilot_train_subjects,
}

with open(PILOT_SUBJECTS_JSON, "w", encoding="utf-8") as f:
    json.dump(pilot_subject_payload, f, indent=2)

print("Saved pilot subject list to:", PILOT_SUBJECTS_JSON)

display(
    pilot_train_df.groupby(["class_name"])
    .size()
    .reset_index(name="count")
    .sort_values("class_name")
)


Saved pilot subject list to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/pilot_train_subjects_subject_class_balanced_20k.json


,class_name,count
0,closed,383
1,open,383


In [10]:
if PILOT_ROOT.exists():
    shutil.rmtree(PILOT_ROOT)

for root in [PILOT_TRAIN_ROOT, PILOT_VAL_ROOT]:
    (root / "open").mkdir(parents=True, exist_ok=True)
    (root / "closed").mkdir(parents=True, exist_ok=True)

print("Created pilot dataset root:", PILOT_ROOT)


Created pilot dataset root: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/pilot_clean_subject_class_balanced_20k


In [11]:
def safe_destination(dst_dir: Path, preferred_name: str, prefix: str) -> Path:
    candidate = dst_dir / preferred_name
    if not candidate.exists():
        return candidate

    stem = Path(preferred_name).stem
    suffix = Path(preferred_name).suffix

    candidate = dst_dir / f"{prefix}__{preferred_name}"
    if not candidate.exists():
        return candidate

    counter = 1
    while True:
        candidate = dst_dir / f"{prefix}__{stem}__dup{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1

pilot_records = []

copy_plan = [
    ("train", pilot_train_df, PILOT_TRAIN_ROOT),
    ("val", pilot_val_df, PILOT_VAL_ROOT),
]

for split_name, df_part, split_root in copy_plan:
    rows = df_part.to_dict("records")
    total = len(rows)

    for i, row in enumerate(rows, 1):
        if i == 1 or i % 1000 == 0:
            print(f"Copying {split_name}: {i}/{total}")

        src_path = Path(row["dest_path"])
        class_name = row["class_name"]
        subject_id = row["subject_id"]
        file_name = row["dest_file_name"]

        assert src_path.exists(), f"Missing source file: {src_path}"

        dst_dir = split_root / class_name
        dst_path = safe_destination(dst_dir, file_name, subject_id)
        shutil.copy2(src_path, dst_path)

        pilot_records.append({
            "split": split_name,
            "class_name": class_name,
            "subject_id": subject_id,
            "source_path": str(src_path),
            "dest_path": str(dst_path),
            "dest_file_name": dst_path.name,
        })

pilot_manifest_df = pd.DataFrame(pilot_records)
pilot_manifest_df.to_csv(PILOT_MANIFEST_CSV, index=False)

print("Saved pilot manifest to:", PILOT_MANIFEST_CSV)
print("Pilot rows:", len(pilot_manifest_df))


Copying train: 1/766
Copying val: 1/1946
Copying val: 1000/1946
Saved pilot manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/pilot_manifest_subject_class_balanced_20k.csv
Pilot rows: 2712


In [12]:
pilot_counts = (
    pilot_manifest_df
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "class_name"])
)

pilot_counts.to_csv(PILOT_COUNTS_CSV, index=False)

display(pilot_counts)
print("Saved pilot counts to:", PILOT_COUNTS_CSV)


,split,class_name,count
0,train,closed,383
1,train,open,383
2,val,closed,973
3,val,open,973


Saved pilot counts to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/pilot_split_counts_subject_class_balanced_20k.csv


In [13]:
print("Pilot train subjects actually materialized:")
display(
    pilot_manifest_df[pilot_manifest_df["split"] == "train"]
    .groupby(["subject_id", "class_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["subject_id", "class_name"])
)

print("Pilot validation class counts:")
display(
    pilot_manifest_df[pilot_manifest_df["split"] == "val"]
    .groupby(["class_name"])
    .size()
    .reset_index(name="count")
    .sort_values("class_name")
)


Pilot train subjects actually materialized:


,subject_id,class_name,count
0,s0002,closed,107
1,s0002,open,107
2,s0003,closed,120
3,s0003,open,120
4,s0005,closed,28
5,s0005,open,28
6,s0007,closed,11
7,s0007,open,11
8,s0009,closed,24
9,s0009,open,24


Pilot validation class counts:


,class_name,count
0,closed,973
1,open,973


In [14]:
!find "{PILOT_TRAIN_ROOT}/open" -type f | wc -l
!find "{PILOT_TRAIN_ROOT}/closed" -type f | wc -l
!find "{PILOT_VAL_ROOT}/open" -type f | wc -l
!find "{PILOT_VAL_ROOT}/closed" -type f | wc -l


383
383
973
973


In [15]:
PILOT_DATA_ROOT = str(PILOT_ROOT)
print(PILOT_DATA_ROOT)


/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/pilot_clean_subject_class_balanced_20k


In [16]:
!python3 /content/Task-driven-low-light-enhancement/train_transfer_detector.py \
  {PILOT_DATA_ROOT} \
  --runtime colab \
  --colab-workspace-root /content/Task-driven-low-light-enhancement \
  --backbone resnet18 \
  --epochs 8 \
  --batch-size 64 \
  --num-workers 2 \
  --learning-rate 1e-4 \
  --monitor f1 \
  --threshold-objective f1 \
  --threshold-candidates 0.2 0.3 0.4 0.5 0.6 0.7 \
  --early-stopping-patience 4 \
  --scheduler-patience 2 \
  --deterministic \
  --save-path artifacts/kaggle_v1_pilot/kaggle_v1_clean_pilot_subject_class_balanced_20k_best.pt \
  --save-last-path artifacts/kaggle_v1_pilot/kaggle_v1_clean_pilot_subject_class_balanced_20k_last.pt \
  --drive-checkpoint-dir {PILOT_CKPT_DIR}


Adjusted initial batch size from 64 to 24 based on available GPU memory.
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /content/Task-driven-low-light-enhancement/artifacts/torch_cache/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 187MB/s]
Runtime: colab
Workspace root: /content/Task-driven-low-light-enhancement
Training on device: cuda
Classes: {'open': 0, 'closed': 1}
Samples: train=766 | val=1946
Focal alpha: [1.0, 1.0]
Best checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_pilot/kaggle_v1_clean_pilot_subject_class_balanced_20k_best.pt
Last checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_pilot/kaggle_v1_clean_pilot_subject_class_balanced_20k_last.pt
Drive backup directory: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/pilot_clean_detector_subject_class_balanced_20k_resnet18
                                                         
Epoch 1/8 | lr=0.000100
Train    

In [17]:
!ls {PILOT_BEST_CKPT}
!ls {PILOT_LAST_CKPT}


/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/pilot_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_pilot_subject_class_balanced_20k_best.pt
/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/pilot_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_pilot_subject_class_balanced_20k_last.pt


In [18]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {PILOT_BEST_CKPT} \
  {VAL_CLEAN_ROOT} \
  --batch-size 64 \
  --num-workers 2 \
  --output-dir {VAL_REPORT_DIR}


Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Clean    loss: 0.0249 | accuracy: 0.9676 | precision: 0.9633 | recall: 0.9723 | f1: 0.9678 | closed_recall: 0.9723 | threshold: 0.50
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 96.76% | 96.33% | 97.23% | 96.78% |

Saved compact CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k/experiment_results.csv
Saved detailed CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k/evaluation_results.csv


In [24]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {PILOT_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --output-dir {TEST_REPORT_DIR}


Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Clean    loss: 0.0445 | accuracy: 0.9194 | precision: 0.8711 | recall: 0.9845 | f1: 0.9243 | closed_recall: 0.9845 | threshold: 0.50
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 91.94% | 87.11% | 98.45% | 92.43% |

Saved compact CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k/experiment_results.csv
Saved detailed CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k/evaluation_results.csv


In [25]:
!sed -n '1,200p' {VAL_REPORT_DIR}/evaluation_summary.txt
!sed -n '1,200p' {TEST_REPORT_DIR}/evaluation_summary.txt


Using threshold=0.50 for the closed-eye class.
Closed-eye recall on clean data: 0.9723.Using threshold=0.50 for the closed-eye class.
Closed-eye recall on clean data: 0.9845.

In [28]:
from pathlib import Path
import pandas as pd

paths = {
    "val": Path("/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k/experiment_results.csv"),
    "test": Path("/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k/experiment_results.csv"),
}

for name, path in paths.items():
    print(f"\n=== {name.upper()} ===")
    print("Path:", path)

    if not path.exists():
        print("Missing file.")
        continue

    df = pd.read_csv(path)
    print(df.to_string(index=False))



=== VAL ===
Path: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k/experiment_results.csv
Dataset  Accuracy  Precision  Recall    F1
  Clean     96.76      96.33   97.23 96.78

=== TEST ===
Path: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k/experiment_results.csv
Dataset  Accuracy  Precision  Recall    F1
  Clean     91.94      87.11   98.45 92.43


In [29]:
print("Phase 3 complete.")
print("PILOT_SUBJECTS_JSON:", PILOT_SUBJECTS_JSON)
print("PILOT_MANIFEST_CSV:", PILOT_MANIFEST_CSV)
print("PILOT_BEST_CKPT:", PILOT_BEST_CKPT)
print("VAL_REPORT_DIR:", VAL_REPORT_DIR)
print("TEST_REPORT_DIR:", TEST_REPORT_DIR)


Phase 3 complete.
PILOT_SUBJECTS_JSON: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/pilot_train_subjects_subject_class_balanced_20k.json
PILOT_MANIFEST_CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/pilot_manifest_subject_class_balanced_20k.csv
PILOT_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/pilot_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_pilot_subject_class_balanced_20k_best.pt
VAL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_val_subject_class_balanced_20k
TEST_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_pilot_clean_on_test_subject_class_balanced_20k
